# Faruq-v3 — capacity-matched multilevel head static audit

Mengaudit wiring MHC0 (P5 control) dan MHF1 (P3+P4+P5 fusion) sebelum training. Tidak membaca dataset atau test dan tidak melakukan optimisasi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'experiments/faruq-v3-fusion-cm512-v1/fusion_cm512.json',
))
AUTHORIZATION = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-fusion-cm512-v1/fusion_cm512.json')
MODEL_YAML = REPO / 'configs/coffee_fg/models/yolo26n-p3.yaml'
WEIGHTS = REPO / 'yolo26n.pt'
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-multilevel-head-v1/static_audit.json'
assert MODEL_YAML.is_file()
assert WEIGHTS.is_file()
print('AUTHORIZATION:', AUTHORIZATION)
print('OUTPUT       :', OUTPUT)

In [ ]:
from coffee_detector.multilevel_head.audit import static_multilevel_head_audit
result = static_multilevel_head_audit(
    MODEL_YAML, OUTPUT, nc=21, weights=WEIGHTS, image_size=128, topk=32
)
assert result['training_executed'] is False
assert result['dataset_accessed'] is False
assert result['test_images_accessed'] is False
print('PARAMETERS:', result['parameter_counts'])
print('ADDED FRACTION:', f"{result['added_parameter_fraction']:.2%}")
print('IDENTITY:', result['identity'])
print('LATENCY:', result['latency_ms_cpu_smoke'])
print('GATES:', result['gates'])
print('DECISION:', result['decision'])
print('NEXT:', result['next_action'])
print('SUMMARY:', result['summary'])
print('Kirim seluruh output ini. Jangan training.')